# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"The total revenue across all orders is ${total_revenue:.2f}.")
print(f"The total number of units sold is {total_units}.")

The total revenue across all orders is $8520.00.
The total number of units sold is 783.


I created the `revenue` column by multiplying `qty` by `price`, then summed the revenue and quantity columns. The 400 orders generated $8,520.00 in revenue from 783 units sold.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).reset_index()
revenue_by_category['share_of_total_revenue'] = (revenue_by_category['revenue'] / total_revenue) * 100
display(revenue_by_category)

,category,revenue,share_of_total_revenue
0,Food,4293.0,50.387324
1,Merch,1771.5,20.792254
2,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


I grouped the orders by `category`, summed the revenue for each category, and calculated each category's percentage of total revenue. Food generated the most revenue at $4,293.00 (50.4%), followed by Merch at 20.8%, Drink at 18.2%, and RainGear at 10.6%.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vendor_summary = df.groupby('vendor_id').agg(
    average_order_revenue=('revenue', 'mean'),
    order_count=('vendor_id', 'count')
).sort_values(by='average_order_revenue', ascending=False).reset_index()

display(vendor_summary)

print("Vendor V-01 has the highest average order revenue.")

,vendor_id,average_order_revenue,order_count
0,V-01,22.595745,94
1,V-18,21.750000,108
2,V-05,20.580645,93
3,V-10,20.314286,105


Vendor V-01 has the highest average order revenue.


I grouped the orders by `vendor_id` and calculated each vendor's average order revenue and number of orders. V-01 had the highest average order revenue at about \$22.60 across 94 orders, followed by V-18 at $21.75 across 108 orders.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
merch_share = (merch_revenue / total_revenue) * 100
print(f"{merch_share:.1f}%")

20.8%


I filtered the data to include only `Merch` orders and divided Merch revenue by total revenue. Merch accounts for 20.8% of the total revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

assert len(joined) == len(df), "Row count changed during merge!"
assert joined['revenue'].sum() == df['revenue'].sum(), "Revenue total changed during merge!"

unmatched_vendors = joined[joined['vendor_name'].isna()]['vendor_id'].unique()

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown Vendor')
display(joined.head())

,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown Vendor
2,V-18,Drink,3,4.5,13.5,Unknown Vendor
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown Vendor


I merged the vendor lookup table into the orders using a left join and verified that neither the row count nor total revenue changed. Vendor V-18 was missing from the lookup, so I labeled it as "Unknown Vendor" instead of removing those orders.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot_report = pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).fillna(0)

display(pivot_report)

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown Vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


I created a pivot table showing revenue for each vendor across the different product categories, with row and column totals included. Unknown Vendor had the highest total revenue at \$2,349.00, while Food generated the most revenue overall at \$4,293.00.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(revenue_by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

### a)
I would recommend that vendors focus more on Food, since it generated \$4,293.00, or about 50.4% of total revenue. They could also consider stocking less RainGear, which only generated \$901.50, and use that space for higher-demand items.

### b)
The least trustworthy result is the vendor comparison because V-18 was missing from the vendor lookup and had to be labeled as "Unknown Vendor."